# 🚀 Advanced Change Detection: Architecture Comparison Study

**Extending our baseline** by comparing **five different architectures** on LEVIR-CD:

| # | Model | Strategy | Key Idea |
|---|-------|----------|----------|
| 1 | Baseline U-Net (ResNet-34) | 6-ch concat | Reference from Notebook 1 |
| 2 | Siamese U-Net (ResNet-34) | Shared encoder | Feature differencing at all scales |
| 3 | Attention-Enhanced (ResNet-34) | Cross-attention | BIT-CD inspired transformer at bottleneck |
| 4 | U-Net + EfficientNet-B3 | 6-ch concat | Lighter, more efficient encoder |
| 5 | U-Net + ResNeXt-50 | 6-ch concat | Stronger encoder with group convolutions |

All models trained under **identical conditions** (same data, hyperparameters, epochs) for fair comparison.

**Relevance to GIM Lab:** Demonstrates understanding of multiple CD architectures, systematic evaluation methodology, and the impact of design choices — key skills for research.

---

## Step 1: Environment Setup

In [ ]:
# Install required packages
!pip install -q segmentation-models-pytorch albumentations huggingface_hub datasets

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm.auto import tqdm
import time
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp
import pandas as pd

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## Step 2: Load LEVIR-CD Dataset

Same dataset as Notebook 1 — LEVIR-CD cropped 256×256 patches from Hugging Face.

In [ ]:
from huggingface_hub import login
from datasets import load_dataset

login(token='YOUR_HF_TOKEN')  # Replace with your Hugging Face token

print('Loading LEVIR-CD dataset...')
dataset = load_dataset('ericyu/LEVIRCD_Cropped256')
print(f'Dataset loaded: {list(dataset.keys())}')
for split in dataset:
    print(f'  {split}: {len(dataset[split])} samples')

In [ ]:
class LEVIRCDDataset(Dataset):
    # PyTorch Dataset for LEVIR-CD change detection.

    def __init__(self, hf_dataset, split='train', img_size=256, augment=False):
        self.data = hf_dataset[split]
        self.img_size = img_size
        self.augment = augment

        if augment:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
                A.GaussNoise(var_limit=(10, 50), p=0.2),
            ], additional_targets={'image_b': 'image'})
        else:
            self.transform = A.Compose([
                A.Resize(img_size, img_size),
            ], additional_targets={'image_b': 'image'})

        self.normalize = A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        img_a = np.array(sample['imageA'].convert('RGB'))
        img_b = np.array(sample['imageB'].convert('RGB'))
        mask = np.array(sample['label'].convert('L'))
        mask = (mask > 127).astype(np.float32)

        augmented = self.transform(image=img_a, image_b=img_b, mask=mask)
        img_a = augmented['image']
        img_b = augmented['image_b']
        mask = augmented['mask']

        norm_a = self.normalize(image=img_a)
        norm_b = self.normalize(image=img_b)

        img_concat = torch.cat([norm_a['image'], norm_b['image']], dim=0)  # [6, H, W]
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).float()  # [1, H, W]

        return img_concat, mask_tensor

print('Dataset class ready')

In [ ]:
# Configuration
IMG_SIZE = 256
BATCH_SIZE = 16
NUM_WORKERS = 0   # Use 0 on Kaggle/Colab
NUM_EPOCHS = 20   # Per model (total: 5 models x 20 = 100 epochs)
LEARNING_RATE = 1e-4

# Create datasets
train_dataset = LEVIRCDDataset(dataset, split='train', img_size=IMG_SIZE, augment=True)
val_dataset = LEVIRCDDataset(dataset, split='val', img_size=IMG_SIZE, augment=False)
test_dataset = LEVIRCDDataset(dataset, split='test', img_size=IMG_SIZE, augment=False)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'Test:  {len(test_dataset)} samples, {len(test_loader)} batches')
print(f'\nTraining plan: {NUM_EPOCHS} epochs/model x 5 models = {NUM_EPOCHS * 5} total epochs')

## Step 3: Model Architectures

We compare five architectures to understand the impact of different design choices for change detection.

### 3a. Baseline: 6-Channel Concatenation U-Net

The simplest approach — concatenate before/after images into a 6-channel input and feed to a standard U-Net. This was our Notebook 1 approach.

In [ ]:
def build_baseline_model(encoder_name='resnet34'):
    # Baseline: 6-channel concatenation U-Net.
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights='imagenet',
        in_channels=6,
        classes=1,
        activation=None,
    )

print('Baseline model builder ready')

### 3b. Siamese U-Net (Shared Encoder + Feature Differencing)

**Key idea:** Instead of concatenating images upfront, process each image through the **same encoder** (weight-sharing) independently. Then compute the **absolute difference** of extracted features at every encoder scale. The decoder learns to interpret these difference features as change maps.

**Advantages:**
- Each image gets proper 3-channel processing (preserves ImageNet pretrained weights perfectly)
- Feature differences capture semantic changes at multiple abstraction levels
- Classic Siamese design used in FC-Siam-Diff and similar papers

In [ ]:
class SiameseUNet(nn.Module):
    # Siamese U-Net with shared encoder and multi-scale feature differencing.

    def __init__(self, encoder_name='resnet34'):
        super().__init__()
        # Build a reference U-Net and extract its components
        ref = smp.Unet(encoder_name, encoder_weights='imagenet',
                       in_channels=3, classes=1, activation=None)
        self.encoder = ref.encoder            # Shared pretrained encoder
        self.decoder = ref.decoder            # Standard U-Net decoder
        self.segmentation_head = ref.segmentation_head

    def forward(self, x):
        # Split 6-channel input into before/after
        img_a = x[:, :3]   # Before image (B, 3, H, W)
        img_b = x[:, 3:]   # After image  (B, 3, H, W)

        # Process both through the SAME encoder (weight sharing)
        feats_a = self.encoder(img_a)  # List of multi-scale features
        feats_b = self.encoder(img_b)

        # Compute absolute difference at every scale
        # This captures "what changed" at each level of abstraction
        diff_feats = []
        for fa, fb in zip(feats_a, feats_b):
            diff_feats.append(torch.abs(fb - fa))

        # Decode difference features into change mask
        decoder_out = self.decoder(diff_feats)
        return self.segmentation_head(decoder_out)

# Verify
_test = SiameseUNet('resnet34')
_out = _test(torch.randn(1, 6, 256, 256))
print(f'Siamese U-Net: input (1, 6, 256, 256) -> output {_out.shape}')
_p = sum(p.numel() for p in _test.parameters()) / 1e6
print(f'  Parameters: {_p:.1f}M')
del _test, _out

### 3c. Attention-Enhanced Change Detector (BIT-CD Inspired)

**Key idea:** Add a **cross-attention transformer block** at the bottleneck of the Siamese encoder. The "after" image features attend to the "before" image features, allowing the model to explicitly learn **which spatial regions changed** through attention weights.

**Inspired by:** BIT-CD (Binary Image Transformer for Change Detection, Chen et al. 2021) — a highly cited paper in the change detection field.

**Architecture:**
1. Shared encoder processes both images independently
2. At the deepest feature level (bottleneck), apply cross-attention with positional embeddings
3. Compute feature differences (attended bottleneck + regular diffs for other scales)
4. Decode into change mask

In [ ]:
class AttentionChangeDetector(nn.Module):
    # Change detector with cross-attention at bottleneck (BIT-CD inspired).

    def __init__(self, encoder_name='resnet34', img_size=256):
        super().__init__()
        # Build reference U-Net and extract components
        ref = smp.Unet(encoder_name, encoder_weights='imagenet',
                       in_channels=3, classes=1, activation=None)
        self.encoder = ref.encoder
        self.decoder = ref.decoder
        self.segmentation_head = ref.segmentation_head

        # Get bottleneck channel count (512 for ResNet-34)
        bottleneck_ch = ref.encoder.out_channels[-1]

        # Spatial resolution at bottleneck: img_size / 32
        spatial_dim = img_size // 32
        num_tokens = spatial_dim * spatial_dim  # 64 for 256x256 input

        # --- Cross-attention transformer block ---
        self.cross_attn = nn.MultiheadAttention(
            bottleneck_ch, num_heads=8, batch_first=True, dropout=0.1
        )
        self.norm1 = nn.LayerNorm(bottleneck_ch)
        self.ffn = nn.Sequential(
            nn.Linear(bottleneck_ch, bottleneck_ch * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(bottleneck_ch * 4, bottleneck_ch),
            nn.Dropout(0.1),
        )
        self.norm2 = nn.LayerNorm(bottleneck_ch)

        # Learnable positional embeddings for spatial features
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens, bottleneck_ch) * 0.02)

    def forward(self, x):
        img_a = x[:, :3]
        img_b = x[:, 3:]

        # Shared encoder
        feats_a = self.encoder(img_a)
        feats_b = self.encoder(img_b)

        # --- Cross-attention at bottleneck ---
        fa = feats_a[-1]   # (B, C, H, W) - deepest features
        fb = feats_b[-1]
        B, C, H, W = fa.shape

        # Reshape to sequences + add positional embeddings
        fa_seq = fa.flatten(2).permute(0, 2, 1) + self.pos_embed   # (B, HW, C)
        fb_seq = fb.flatten(2).permute(0, 2, 1) + self.pos_embed

        # Cross-attention: "after" attends to "before" -> highlights what changed
        attn_out, _ = self.cross_attn(fb_seq, fa_seq, fa_seq)
        attn_out = self.norm1(attn_out + fb_seq)        # Residual
        attn_out = self.norm2(self.ffn(attn_out) + attn_out)  # FFN + residual

        # Reshape back to spatial feature map
        fb_attended = attn_out.permute(0, 2, 1).view(B, C, H, W)

        # Build difference features for decoder
        diff_feats = []
        for i, (fa_i, fb_i) in enumerate(zip(feats_a, feats_b)):
            if i == len(feats_a) - 1:
                # Bottleneck: use attention-enhanced features
                diff_feats.append(torch.abs(fb_attended - fa))
            else:
                # Other scales: standard absolute difference
                diff_feats.append(torch.abs(fb_i - fa_i))

        decoder_out = self.decoder(diff_feats)
        return self.segmentation_head(decoder_out)

# Verify
_test = AttentionChangeDetector('resnet34', img_size=256)
_out = _test(torch.randn(1, 6, 256, 256))
print(f'Attention Change Detector: input (1, 6, 256, 256) -> output {_out.shape}')
_p = sum(p.numel() for p in _test.parameters()) / 1e6
_attn_p = sum(p.numel() for n, p in _test.named_parameters()
              if any(k in n for k in ['cross_attn', 'ffn', 'norm', 'pos_embed']))
print(f'  Total parameters: {_p:.1f}M (attention overhead: {_attn_p/1e6:.1f}M)')
del _test, _out

## Step 4: Training Infrastructure

Same loss function, metrics, and training loop as Notebook 1 for a fair comparison.

In [ ]:
class BCEDiceLoss(nn.Module):
    # Combined Binary Cross-Entropy and Dice Loss.
    def __init__(self, bce_weight=0.5, dice_weight=0.5, smooth=1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred_sigmoid = torch.sigmoid(pred)
        pred_flat = pred_sigmoid.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = (2. * intersection + self.smooth) / (pred_flat.sum() + target_flat.sum() + self.smooth)
        dice_loss = 1 - dice
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss


class ChangeDetectionMetrics:
    # Compute F1, Precision, Recall, IoU for change detection.
    def __init__(self, threshold=0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        pred_binary = (torch.sigmoid(pred) > self.threshold).float()
        target_binary = target.float()
        self.tp += ((pred_binary == 1) & (target_binary == 1)).sum().item()
        self.fp += ((pred_binary == 1) & (target_binary == 0)).sum().item()
        self.fn += ((pred_binary == 0) & (target_binary == 1)).sum().item()
        self.tn += ((pred_binary == 0) & (target_binary == 0)).sum().item()

    def compute(self):
        precision = self.tp / (self.tp + self.fp + 1e-8)
        recall = self.tp / (self.tp + self.fn + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        iou = self.tp / (self.tp + self.fp + self.fn + 1e-8)
        oa = (self.tp + self.tn) / (self.tp + self.tn + self.fp + self.fn + 1e-8)
        return {'F1': f1, 'Precision': precision, 'Recall': recall, 'IoU': iou, 'OA': oa}

print('Loss and metrics ready')

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    metrics = ChangeDetectionMetrics()

    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        metrics.update(outputs.detach(), masks)
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    return running_loss / len(loader), metrics.compute()


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch, phase='Val'):
    model.eval()
    running_loss = 0.0
    metrics = ChangeDetectionMetrics()

    pbar = tqdm(loader, desc=f'Epoch {epoch} [{phase}]', leave=False)
    for imgs, masks in pbar:
        imgs = imgs.to(device)
        masks = masks.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        running_loss += loss.item()
        metrics.update(outputs, masks)

    return running_loss / len(loader), metrics.compute()

print('Training functions ready')

In [ ]:
def train_model(model_name, model, train_loader, val_loader, device,
                num_epochs=20, lr=1e-4):
    # Train a single model and return results.
    model = model.to(device)
    criterion = BCEDiceLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=1e-6
    )

    params_m = sum(p.numel() for p in model.parameters()) / 1e6

    print(f"\n{'='*60}")
    print(f"  Training: {model_name} ({params_m:.1f}M parameters)")
    print(f"{'='*60}")

    best_val_f1 = 0.0
    best_epoch = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_f1': [], 'val_f1': [],
        'train_iou': [], 'val_iou': []
    }

    for epoch in range(1, num_epochs + 1):
        train_loss, train_scores = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch
        )
        val_loss, val_scores = evaluate(
            model, val_loader, criterion, device, epoch
        )
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_f1'].append(train_scores['F1'])
        history['val_f1'].append(val_scores['F1'])
        history['train_iou'].append(train_scores['IoU'])
        history['val_iou'].append(val_scores['IoU'])

        if val_scores['F1'] > best_val_f1:
            best_val_f1 = val_scores['F1']
            best_epoch = epoch
            torch.save(model.state_dict(), f'best_{model_name}.pth')
            marker = ' *BEST*'
        else:
            marker = ''

        print(f"  Epoch {epoch:02d}/{num_epochs} | "
              f"Train F1: {train_scores['F1']:.4f} | "
              f"Val F1: {val_scores['F1']:.4f}, "
              f"IoU: {val_scores['IoU']:.4f}{marker}")

    # Load best weights
    model.load_state_dict(torch.load(f'best_{model_name}.pth', map_location=device))
    print(f"  Best model: epoch {best_epoch}, Val F1: {best_val_f1:.4f}")

    return model, history, {
        'best_f1': best_val_f1,
        'best_epoch': best_epoch,
        'params_m': params_m
    }


def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    # Denormalize a tensor image for visualization.
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)

print('Training wrapper ready')

## Step 5: Train & Evaluate All Models

Training all 5 models sequentially. Each model:
- Trains for 20 epochs with AdamW + CosineAnnealingLR
- Saves best checkpoint (by Val F1)
- Evaluates on test set after training
- Saves predictions for visual comparison

**If a checkpoint exists**, the model is loaded instead of re-trained (safe for re-runs).

> ⏱ **Estimated time:** ~2-3 hours total on T4 GPU (Kaggle free tier)

In [ ]:
# Models to train and compare
models_config = [
    ('Baseline_ResNet34',      lambda: build_baseline_model('resnet34')),
    ('Siamese_ResNet34',       lambda: SiameseUNet('resnet34')),
    ('Attention_ResNet34',     lambda: AttentionChangeDetector('resnet34', img_size=IMG_SIZE)),
    ('Baseline_EfficientNetB3', lambda: build_baseline_model('efficientnet-b3')),
    ('Baseline_ResNeXt50',     lambda: build_baseline_model('resnext50_32x4d')),
]

# Fixed test indices for visual comparison across all models
random.seed(42)
VIS_INDICES = sorted(random.sample(range(len(test_dataset)), 6))
print(f'Visualization indices: {VIS_INDICES}')
print(f'Models to train: {len(models_config)}')
print(f'Epochs per model: {NUM_EPOCHS}')
print(f'Total epochs: {len(models_config) * NUM_EPOCHS}')

In [ ]:
# ========== MAIN TRAINING LOOP ==========
all_test_results = {}
all_histories = {}
all_predictions = {}
all_train_times = {}

for model_name, model_fn in models_config:
    start_time = time.time()
    checkpoint_path = f'best_{model_name}.pth'

    # Build model
    model = model_fn()

    # Train or load checkpoint
    if os.path.exists(checkpoint_path):
        model = model.to(DEVICE)
        model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
        print(f"\nLoaded {model_name} from checkpoint")
        all_histories[model_name] = None
    else:
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        random.seed(SEED)
        model, history, info = train_model(
            model_name, model, train_loader, val_loader, DEVICE,
            num_epochs=NUM_EPOCHS, lr=LEARNING_RATE
        )
        all_histories[model_name] = history

    elapsed = (time.time() - start_time) / 60
    all_train_times[model_name] = elapsed

    # Evaluate on test set
    criterion = BCEDiceLoss()
    test_loss, test_scores = evaluate(model, test_loader, criterion, DEVICE, 0, 'Test')
    test_scores['Loss'] = round(test_loss, 4)
    test_scores['Params_M'] = round(sum(p.numel() for p in model.parameters()) / 1e6, 1)
    test_scores['Time_min'] = round(elapsed, 1)
    all_test_results[model_name] = test_scores

    # Save predictions for visualization
    model.eval()
    preds = {}
    with torch.no_grad():
        for idx in VIS_INDICES:
            imgs, mask = test_dataset[idx]
            pred = torch.sigmoid(model(imgs.unsqueeze(0).to(DEVICE))).cpu().squeeze().numpy()
            preds[idx] = (pred > 0.5).astype(np.float32)
    all_predictions[model_name] = preds

    print(f"  Test F1={test_scores['F1']:.4f}, IoU={test_scores['IoU']:.4f} ({elapsed:.1f} min)")

    # Free GPU memory
    del model
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("All models trained and evaluated!")
print(f"{'='*60}")

## Step 6: Results Comparison

Quantitative comparison of all five architectures on the LEVIR-CD test set.

In [ ]:
# Create comparison DataFrame
results_df = pd.DataFrame(all_test_results).T
results_df.index.name = 'Model'

# Reorder columns
cols = ['F1', 'IoU', 'Precision', 'Recall', 'OA', 'Params_M', 'Time_min', 'Loss']
results_df = results_df[[c for c in cols if c in results_df.columns]]
results_df = results_df.sort_values('F1', ascending=False)

# Display formatted
print("=" * 85)
print("                    MODEL COMPARISON - TEST SET RESULTS")
print("=" * 85)
display_df = results_df.copy()
for col in ['F1', 'IoU', 'Precision', 'Recall', 'OA', 'Loss']:
    if col in display_df.columns:
        display_df[col] = display_df[col].apply(lambda x: f'{float(x):.4f}')
if 'Params_M' in display_df.columns:
    display_df['Params_M'] = display_df['Params_M'].apply(lambda x: f'{float(x):.1f}')
if 'Time_min' in display_df.columns:
    display_df['Time_min'] = display_df['Time_min'].apply(lambda x: f'{float(x):.1f}')
print(display_df.to_string())

best_model = results_df.index[0]
print(f"\nBest model: {best_model} (F1 = {float(results_df.loc[best_model, 'F1']):.4f})")

# Bar chart comparison
results_num = results_df[['F1', 'IoU']].astype(float).sort_values('F1', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']
colors_sorted = colors[:len(results_num)]

results_num['F1'].plot(kind='barh', ax=axes[0], color=colors_sorted, edgecolor='white')
axes[0].set_title('F1 Score Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('F1 Score')
for i, v in enumerate(results_num['F1']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10, fontweight='bold')

results_num['IoU'].plot(kind='barh', ax=axes[1], color=colors_sorted, edgecolor='white')
axes[1].set_title('IoU Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('IoU')
for i, v in enumerate(results_num['IoU']):
    axes[1].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=10, fontweight='bold')

plt.suptitle('Architecture Comparison on LEVIR-CD Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7: Training Curves Comparison

In [ ]:
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for i, (name, hist) in enumerate(all_histories.items()):
    if hist is None:
        continue
    epochs = range(1, len(hist['val_f1']) + 1)
    axes[0].plot(epochs, hist['val_f1'], color=colors[i % len(colors)],
                 label=name.replace('_', ' '), linewidth=2)
    axes[1].plot(epochs, hist['val_iou'], color=colors[i % len(colors)],
                 label=name.replace('_', ' '), linewidth=2)
    axes[2].plot(epochs, hist['val_loss'], color=colors[i % len(colors)],
                 label=name.replace('_', ' '), linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation F1')
axes[0].set_title('Validation F1 Score', fontsize=13)
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation IoU')
axes[1].set_title('Validation IoU', fontsize=13)
axes[1].legend(fontsize=8, loc='lower right')
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Validation Loss')
axes[2].set_title('Validation Loss', fontsize=13)
axes[2].legend(fontsize=8, loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Training Curves - All Architectures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8: Visual Comparison

Side-by-side predictions from all models on the same test samples.

In [ ]:
model_names = list(all_predictions.keys())
num_models = len(model_names)
num_samples = len(VIS_INDICES)

fig, axes = plt.subplots(num_samples, num_models + 3,
                         figsize=(3 * (num_models + 3), 3 * num_samples))

for i, idx in enumerate(VIS_INDICES):
    imgs, mask = test_dataset[idx]
    img_a = denormalize(imgs[:3]).permute(1, 2, 0).numpy()
    img_b = denormalize(imgs[3:]).permute(1, 2, 0).numpy()
    gt = mask.squeeze().numpy()

    axes[i, 0].imshow(img_a)
    axes[i, 0].set_title('Before' if i == 0 else '', fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_b)
    axes[i, 1].set_title('After' if i == 0 else '', fontsize=10)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(gt, cmap='hot')
    axes[i, 2].set_title('Ground Truth' if i == 0 else '', fontsize=10)
    axes[i, 2].axis('off')

    for j, name in enumerate(model_names):
        pred = all_predictions[name][idx]
        axes[i, 3 + j].imshow(pred, cmap='hot')
        if i == 0:
            short = name.replace('Baseline_', 'B:').replace('Siamese_', 'S:').replace('Attention_', 'A:')
            axes[i, 3 + j].set_title(short, fontsize=9)
        axes[i, 3 + j].axis('off')

plt.suptitle('Prediction Comparison Across All Architectures',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('model_comparison_visual.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9: Efficiency Analysis

Compare model size, training time, and performance trade-offs.

In [ ]:
# Efficiency scatter plot: F1 vs Parameters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

f1_vals = [float(all_test_results[n]['F1']) for n in all_test_results]
iou_vals = [float(all_test_results[n]['IoU']) for n in all_test_results]
param_vals = [float(all_test_results[n]['Params_M']) for n in all_test_results]
time_vals = [float(all_test_results[n]['Time_min']) for n in all_test_results]
names = list(all_test_results.keys())
colors_scatter = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2', '#59a14f']

for i, name in enumerate(names):
    axes[0].scatter(param_vals[i], f1_vals[i], s=200, c=colors_scatter[i],
                    edgecolor='black', linewidth=1.5, zorder=5)
    axes[0].annotate(name.replace('Baseline_', 'B:').replace('Siamese_', 'S:').replace('Attention_', 'A:'),
                     (param_vals[i], f1_vals[i]), textcoords="offset points",
                     xytext=(10, 5), fontsize=9)

axes[0].set_xlabel('Parameters (M)', fontsize=12)
axes[0].set_ylabel('Test F1 Score', fontsize=12)
axes[0].set_title('F1 Score vs Model Size', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

for i, name in enumerate(names):
    axes[1].scatter(time_vals[i], f1_vals[i], s=200, c=colors_scatter[i],
                    edgecolor='black', linewidth=1.5, zorder=5)
    axes[1].annotate(name.replace('Baseline_', 'B:').replace('Siamese_', 'S:').replace('Attention_', 'A:'),
                     (time_vals[i], f1_vals[i]), textcoords="offset points",
                     xytext=(10, 5), fontsize=9)

axes[1].set_xlabel('Training Time (min)', fontsize=12)
axes[1].set_ylabel('Test F1 Score', fontsize=12)
axes[1].set_title('F1 Score vs Training Time', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Efficiency Analysis: Performance vs Cost', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('efficiency_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 10: Conclusions & Key Findings

### Architecture Comparison Summary

| Aspect | Finding |
|--------|---------|
| **Baseline (6-ch concat)** | Simple, effective, fast to train. Good starting point. |
| **Siamese U-Net** | Better utilizes pretrained weights. Feature differencing is principled. |
| **Attention-Enhanced** | Cross-attention captures spatial change relationships explicitly. Higher capacity. |
| **EfficientNet-B3** | Fewer parameters, potentially better generalization. |
| **ResNeXt-50** | Largest model. Multi-path aggregation may help with complex changes. |

### Key Takeaways

1. **Architecture matters for change detection** — the way we process bi-temporal inputs significantly impacts results
2. **Siamese designs** that preserve 3-channel pretrained weights often outperform naive concatenation
3. **Attention mechanisms** add a small parameter overhead but can meaningfully improve spatial reasoning
4. **Encoder choice** affects both accuracy and efficiency — larger \u2260 always better

### Direct Relevance to GIM Lab Research
- **Urban monitoring**: All models detect building-level changes from satellite imagery
- **Architecture design**: Systematic comparison methodology applicable to any RS task
- **Scalability**: Understanding speed/accuracy trade-offs for real-world deployment
- **Foundation for 3D**: These 2D CD techniques can be extended to LiDAR point cloud change detection

### Next Steps
1. **Ensemble** top-performing models for even higher accuracy
2. **Multi-task learning**: Combine CD with building segmentation
3. **3D extension**: Apply similar architectures to airborne LiDAR data (Prof. Li's core expertise)
4. **Temporal modeling**: Extend to multi-date sequences using recurrent or temporal attention

In [ ]:
# Save all results to JSON
results_export = {
    'project': 'Advanced Change Detection Architecture Comparison',
    'dataset': 'LEVIR-CD (ericyu/LEVIRCD_Cropped256)',
    'epochs_per_model': NUM_EPOCHS,
    'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'optimizer': 'AdamW (lr=1e-4, wd=1e-4)',
    'scheduler': 'CosineAnnealingLR',
    'loss': 'BCE + Dice (0.5 + 0.5)',
    'models': {}
}

for name, scores in all_test_results.items():
    model_results = {}
    for k, v in scores.items():
        if isinstance(v, (int, float, np.integer, np.floating)):
            model_results[k] = round(float(v), 4)
        else:
            model_results[k] = v
    results_export['models'][name] = model_results

with open('comparison_results.json', 'w') as f:
    json.dump(results_export, f, indent=2)

print('Results saved to comparison_results.json')
print()

# Final summary
print('=' * 60)
print('   FINAL SUMMARY')
print('=' * 60)
sorted_models = sorted(all_test_results.items(), key=lambda x: float(x[1]['F1']), reverse=True)
for rank, (name, scores) in enumerate(sorted_models, 1):
    print(f'  #{rank} {name}')
    print(f'      F1={float(scores["F1"]):.4f}  IoU={float(scores["IoU"]):.4f}  '
          f'Params={float(scores["Params_M"]):.1f}M')
print('=' * 60)
print()
print('Use these results in your email to Professor Li!')
print('Highlight the best model and discuss WHY it performed best.')